In [ ]:
import pandas as pd
import numpy as np
import os
import time
import tracemalloc

tracemalloc.start()

start_time = time.time()

class AttackSpecificDatasetGenerator:
    def __init__(self, training_file, testing_file, attack_vectors_file=None, label_column=None,
                 train_contamination_ratio=0.5, test_contamination_ratio=None,
                 train_random_seed=None, test_random_seed=None):
        self.training_data = pd.read_excel(training_file)
        self.testing_data = pd.read_excel(testing_file)

        self.attack_vectors = None
        if attack_vectors_file and os.path.exists(attack_vectors_file):
            try:
                self.attack_vectors = pd.read_excel(attack_vectors_file)
                print(f"Loaded attack vectors from {attack_vectors_file}")
            except Exception as e:
                print(f"Warning: Could not load attack vectors file: {str(e)}")

        self.training_data = self.training_data.apply(
          lambda x: pd.to_numeric(x, errors='coerce').fillna(x) if x.dtype == object else x
           )
        self.testing_data = self.testing_data.apply(
          lambda x: pd.to_numeric(x, errors='coerce').fillna(x) if x.dtype == object else x
            )

        self.label_column = label_column if label_column else "label"
        if self.label_column not in self.training_data.columns:
            self.training_data[self.label_column] = 0
        if self.label_column not in self.testing_data.columns:
            self.testing_data[self.label_column] = 0

        if "attack_type" not in self.training_data.columns:
            self.training_data["attack_type"] = "none"
        if "attack_type" not in self.testing_data.columns:
            self.testing_data["attack_type"] = "none"

        self.train_features = self.training_data.drop(
            columns=[self.label_column, "attack_type"])
        self.test_features = self.testing_data.drop(
            columns=[self.label_column, "attack_type"])
        self.feature_names = self.train_features.columns.tolist()

        self.original_train_data = self.training_data.copy()
        self.original_test_data = self.testing_data.copy()

        self.replay_sources = {}

        self.train_contamination_ratio = train_contamination_ratio

        self.test_contamination_ratio = (
            test_contamination_ratio if test_contamination_ratio is not None
            else train_contamination_ratio
        )

        # --- SEEDS SEPARES POUR TRAIN ET TEST ---
        self.train_random_seed = train_random_seed
        self.test_random_seed = test_random_seed

        # Un RandomState dedie par source de donnees, cree une seule fois
        # (au lieu d'etre recree a chaque appel comme dans la version originale)
        self.rng_by_source = {
            'train': np.random.RandomState(self.train_random_seed),
            'test': np.random.RandomState(self.test_random_seed)
        }

    def _get_rng(self, data_source):
        return self.rng_by_source[data_source]

    def _ensure_numeric(self, z):
        return np.array(z, dtype=np.float64)

    def false_data_injection_attack(self, z, sample_idx=0, data_source='train'):
        if self.attack_vectors is not None and not self.attack_vectors.empty:
            try:
                vector_idx = sample_idx % len(self.attack_vectors)
                attack_vector = self.attack_vectors.iloc[vector_idx].values

                if len(attack_vector) == len(z):
                    return z + attack_vector.astype(np.float64)
                elif len(attack_vector) > len(z):
                    return z + attack_vector[:len(z)].astype(np.float64)
                else:
                    padded_vector = np.zeros(len(z))
                    padded_vector[:len(attack_vector)] = attack_vector
                    return z + padded_vector.astype(np.float64)

            except Exception as e:
                print(f"Warning: Could not apply attack vector, using fallback method: {str(e)}")
                return self._fallback_false_injection(z, data_source)
        else:
            print("Warning: No attack vectors file provided, using fallback method")
            return self._fallback_false_injection(z, data_source)

    def _fallback_false_injection(self, z, data_source='train'):
        rng = self._get_rng(data_source)
        noise_magnitude = 0.1 * np.std(z) if np.std(z) > 0 else 0.1
        return z + rng.normal(0, noise_magnitude, size=len(z))

    def random_replay_attack(self, sequence_idx, offset, window_size, data_source='train'):
        features = self.train_features if data_source == 'train' else self.test_features
        rng = self._get_rng(data_source)

        sequence_key = f"{data_source}_random_replay_{sequence_idx}"

        if sequence_key not in self.replay_sources:
            source_start = rng.randint(0, max(1, len(features) - window_size))
            source_end = min(source_start + window_size - 1, len(features) - 1)

            source_sequence = []
            for i in range(source_start, source_end + 1):
                source_sequence.append(features.iloc[i].values.copy())

            self.replay_sources[sequence_key] = {
                'sequence': source_sequence,
                'start_idx': source_start,
                'end_idx': source_end
            }
            print(f"Created new random replay sequence {sequence_idx} from source [{source_start}-{source_end}]")

        source_data = self.replay_sources[sequence_key]
        if offset < len(source_data['sequence']):
            return source_data['sequence'][offset]
        else:
            return source_data['sequence'][0]

    def one_step_replay_attack(self, z, t, data_source='train'):
        features = self.train_features if data_source == 'train' else self.test_features
        if t <= 0 or len(features) <= 1:
            return z
        idx = max(0, t - 1)
        idx = min(idx, len(features) - 1)
        return features.iloc[idx].values.astype(np.float64)

    def interval_replay_attack(self, sequence_idx, offset, window_size, data_source='train'):
        features = self.train_features if data_source == 'train' else self.test_features
        rng = self._get_rng(data_source)

        sequence_key = f"{data_source}_interval_replay_{sequence_idx}"

        if sequence_key not in self.replay_sources:
            max_start = len(features) - window_size
            source_start = rng.randint(0, max(1, max_start // 2))
            source_end = min(source_start + window_size - 1, len(features) - 1)

            source_sequence = []
            for i in range(source_start, source_end + 1):
                source_sequence.append(features.iloc[i].values.copy())

            self.replay_sources[sequence_key] = {
                'sequence': source_sequence,
                'start_idx': source_start,
                'end_idx': source_end
            }
            print(f"Created new interval replay sequence {sequence_idx} from source [{source_start}-{source_end}]")

        source_data = self.replay_sources[sequence_key]
        if offset < len(source_data['sequence']):
            return source_data['sequence'][offset]
        else:
            return source_data['sequence'][0]

    def apply_attack(self, attack_type, params, data_source='train'):
        features = self.train_features if data_source == 'train' else self.test_features

        if attack_type == 'random_replay':
            return self.random_replay_attack(
                params['sequence_idx'],
                params['offset'],
                params['window_size'],
                data_source
            )

        elif attack_type == 'interval_replay':
            return self.interval_replay_attack(
                params['sequence_idx'],
                params['offset'],
                params['window_size'],
                data_source
            )

        elif attack_type == 'one_step_replay':
            return self.one_step_replay_attack(
                params['original_sample'],
                params['idx'],
                data_source
            )

        elif attack_type == 'false_injection':
            return self.false_data_injection_attack(
                params['original_sample'],
                params.get('sample_idx', 0),
                data_source
            )

        else:
            print(f"Warning: Unknown attack type {attack_type}, returning original sample")
            return params['original_sample']

    def generate_attack_specific_datasets(self, output_dir="/results", window_size=5):
        os.makedirs(output_dir, exist_ok=True)

        attack_config = {
            'false_injection': 0,
            'random_replay': window_size,
            'one_step_replay': 1,
            'interval_replay': window_size
        }

        print(f"Generating datasets for {len(attack_config)} attack types:")
        for attack_type in attack_config.keys():
            print(f"  - {attack_type}")
        print(f"Train contamination ratio: {self.train_contamination_ratio*100:.1f}%")
        print(f"Test contamination ratio: {self.test_contamination_ratio*100:.1f}%")
        print(f"Train random seed: {self.train_random_seed}")
        print(f"Test random seed: {self.test_random_seed}")

        for attack_type in attack_config.keys():
            print(f"\n{'='*50}")
            print(f"Generating datasets for {attack_type} attack")
            print(f"{'='*50}")

            self.training_data = self.original_train_data.copy()
            self.testing_data = self.original_test_data.copy()

            self.train_features = self.training_data.drop(
                columns=[self.label_column, "attack_type"])
            self.test_features = self.testing_data.drop(
                columns=[self.label_column, "attack_type"])

            self.replay_sources = {}

            # On reinitialise les RNG a chaque type d'attaque pour garantir
            # la reproductibilite independante entre chaque run d'attaque
            self.rng_by_source = {
                'train': np.random.RandomState(self.train_random_seed),
                'test': np.random.RandomState(self.test_random_seed)
            }

            for df in [self.training_data, self.testing_data]:
                if "replay_source" not in df.columns:
                    df["replay_source"] = ""
                if "replay_interval" not in df.columns:
                    df["replay_interval"] = -1
                if "replay_offset" not in df.columns:
                    df["replay_offset"] = -1
                if "sequence_id" not in df.columns:
                    df["sequence_id"] = -1

            self._apply_specific_attack_to_dataset('train', attack_type, attack_config[attack_type], window_size)
            self._apply_specific_attack_to_dataset('test', attack_type, attack_config[attack_type], window_size)
            self._save_attack_specific_datasets(attack_type, output_dir)

        return {"status": "All attack-specific datasets generated successfully", "attacks": list(attack_config.keys())}

    def _apply_specific_attack_to_dataset(self, data_source, attack_type, req_window, window_size):
        print(f"\nApplying {attack_type} attack to {data_source} dataset...")

        if data_source == 'train':
            data = self.training_data
            features = self.train_features
        else:
            data = self.testing_data
            features = self.test_features

        total_samples = len(features)

        active_ratio = self.train_contamination_ratio if data_source == 'train' else self.test_contamination_ratio
        num_attack_samples = int(total_samples * active_ratio)

        # Utilise le RNG dedie a cette source (train ou test), avec son propre seed
        rng = self._get_rng(data_source)

        if req_window <= 1:

            all_indices = np.arange(total_samples)
            rng.shuffle(all_indices)
            attack_range = sorted(all_indices[:num_attack_samples].tolist())

        else:

            num_blocks_needed = max(1, num_attack_samples // req_window)
            chosen = set()
            attempts = 0
            max_attempts = num_blocks_needed * 200

            while len(chosen) < num_blocks_needed * req_window and attempts < max_attempts:
                attempts += 1
                start = rng.randint(0, total_samples - req_window + 1)
                block = set(range(start, start + req_window))
                if block & chosen:
                    continue
                chosen |= block

            attack_range = sorted(chosen)

        normal_range = [i for i in range(total_samples) if i not in set(attack_range)]

        data['attack_type'] = 'none'
        data[self.label_column] = 0

        sequence_counter = 0
        remaining_indices = attack_range.copy()

        if req_window > 0:
            while len(remaining_indices) >= req_window:
                start_idx = remaining_indices[0]
                sequence_indices = []

                consecutive_count = 0
                i = 0
                while consecutive_count < req_window and i < len(remaining_indices):
                    current_idx = remaining_indices[i]
                    if current_idx == start_idx + consecutive_count:
                        sequence_indices.append(current_idx)
                        consecutive_count += 1
                        i += 1
                    else:
                        sequence_indices = [current_idx]
                        start_idx = current_idx
                        consecutive_count = 1
                        i += 1

                if len(sequence_indices) == req_window:
                    for idx in sequence_indices:
                        remaining_indices.remove(idx)

                    sequence_counter += 1

                    if attack_type in ['random_replay', 'interval_replay']:
                        sequence_key = f"{data_source}_{attack_type}_{sequence_counter}"

                        for offset, idx in enumerate(sequence_indices):
                            original_sample = features.iloc[idx].values.copy()

                            params = {
                                'original_sample': original_sample,
                                'idx': idx,
                                'sequence_idx': sequence_counter,
                                'offset': offset,
                                'window_size': req_window,
                                'sample_idx': idx
                            }

                            attacked_sample = self.apply_attack(attack_type, params, data_source)

                            feature_cols = features.columns
                            for j, col in enumerate(feature_cols):
                                data.at[idx, col] = attacked_sample[j]

                            data.at[idx, 'attack_type'] = attack_type
                            data.at[idx, self.label_column] = 1
                            data.at[idx, 'sequence_id'] = sequence_counter

                            if sequence_key in self.replay_sources:
                                source_info = self.replay_sources[sequence_key]
                                source_range = f"[{source_info['start_idx']}-{source_info['end_idx']}]"
                                data.at[idx, 'replay_source'] = source_range
                                data.at[idx, 'replay_interval'] = sequence_counter
                                data.at[idx, 'replay_offset'] = offset

                    elif attack_type == 'one_step_replay':
                        for i, idx in enumerate(sequence_indices):
                            original_sample = features.iloc[idx].values.copy()

                            params = {
                                'original_sample': original_sample,
                                'idx': idx,
                                'sample_idx': idx
                            }

                            attacked_sample = self.apply_attack(attack_type, params, data_source)

                            feature_cols = features.columns
                            for j, col in enumerate(feature_cols):
                                data.at[idx, col] = attacked_sample[j]

                            data.at[idx, 'attack_type'] = attack_type
                            data.at[idx, self.label_column] = 1
                            data.at[idx, 'sequence_id'] = sequence_counter

                            source_idx = max(0, idx - 1)
                            data.at[idx, 'replay_source'] = f"[{source_idx}]"
                            data.at[idx, 'replay_interval'] = sequence_counter
                            data.at[idx, 'replay_offset'] = i % req_window
                else:
                    break
        else:
            for idx in attack_range:
                original_sample = features.iloc[idx].values.copy()

                params = {
                    'original_sample': original_sample,
                    'idx': idx,
                    'sample_idx': idx
                }

                attacked_sample = self.apply_attack(attack_type, params, data_source)

                feature_cols = features.columns
                for j, col in enumerate(feature_cols):
                    data.at[idx, col] = attacked_sample[j]

                data.at[idx, 'attack_type'] = attack_type
                data.at[idx, self.label_column] = 1

                if idx % 50 == 0:
                    print(f"Applied {attack_type} attack to sample {idx}")

        attack_count = sum(data['attack_type'] == attack_type)
        normal_count = sum(data['attack_type'] == 'none')
        print(f"\nDataset summary for {data_source}:")
        print(f"Normal samples: {normal_count} ({normal_count/len(data)*100:.1f}%)")
        print(f"{attack_type} attack samples: {attack_count} ({attack_count/len(data)*100:.1f}%)")

        if data_source == 'train':
            self.train_features = self.training_data.drop(
                columns=[self.label_column, "attack_type"])
        else:
            self.test_features = self.testing_data.drop(
                columns=[self.label_column, "attack_type"])

    def _save_attack_specific_datasets(self, attack_type, output_dir):
        attack_name = attack_type.replace('_', '-')

        train_filename = f"{output_dir}/train_{attack_name}.xlsx"
        self.training_data.to_excel(train_filename, index=False)
        print(f"Saved training dataset to {train_filename}")

        test_filename = f"{output_dir}/test_{attack_name}.xlsx"
        self.testing_data.to_excel(test_filename, index=False)
        print(f"Saved testing dataset to {test_filename}")

    def _generate_attack_visualization(self, attack_type, output_dir):
        pass


if __name__ == "__main__":
    try:
        generator = AttackSpecificDatasetGenerator(
            '/content/4_bus_Benign_training_set.xlsx',
            '/content/4_bus_Benign_testing_set.xlsx',
            '',
            train_contamination_ratio=0.10,
            test_contamination_ratio=0.2,
            train_random_seed=123,
            test_random_seed=124
        )

        results = generator.generate_attack_specific_datasets(output_dir="content/results", window_size=5)

        print("\nAttack-Specific Dataset Generation Complete!")
        print(f"Results: {results}")

    except Exception as e:
        print(f"Error: {str(e)}")
end_time = time.time()
current, peak = tracemalloc.get_traced_memory()
tracemalloc.stop()

print(f"Total runtime: {end_time - start_time:.2f} seconds")
print(f"Peak memory: {peak / (1024**2):.2f} MB")

Generating datasets for 4 attack types:
  - false_injection
  - random_replay
  - one_step_replay
  - interval_replay
Train contamination ratio: 10.0%
Test contamination ratio: 20.0%
Train random seed: 48
Test random seed: 124

Generating datasets for false_injection attack

Applying false_injection attack to train dataset...
Applied false_injection attack to sample 200
Applied false_injection attack to sample 1700
Applied false_injection attack to sample 2250
Applied false_injection attack to sample 4050
Applied false_injection attack to sample 4350
Applied false_injection attack to sample 5300
Applied false_injection attack to sample 5400
Applied false_injection attack to sample 5550
Applied false_injection attack to sample 5700
Applied false_injection attack to sample 5850
Applied false_injection attack to sample 7100
Applied false_injection attack to sample 7250
Applied false_injection attack to sample 7500
Applied false_injection attack to sample 7550
Applied false_injection attac